# 02 - Feature Engineering and Tracking (NASA C-MAPSS)

This notebook covers Phase 2 of the implementation plan:
- Rolling window feature transformations (mean, std, lag)
- 3D sequence dataset construction for PyTorch models
- MLflow logging for data versioning and preprocessing parameters

## Colab setup

If running in Google Colab, mount Drive and install dependencies before executing the notebook cells.

Example:
```bash
pip install -r requirements.txt
```

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing src/ and data/.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/Users/ayushkanyal/Desktop/Predictive Maintainence')

In [2]:
import mlflow
import numpy as np
import pandas as pd

from src.data.constants import RECOMMENDED_DROPPED_SENSORS
from src.features import (
    build_rolling_features,
    create_sequence_dataset,
    dataframe_fingerprint,
    get_sensor_columns,
)

pd.set_option("display.max_columns", 120)

/Users/ayushkanyal/Desktop/Predictive Maintainence/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
INPUT_PATH = PROCESSED_DIR / "train_FD001_with_rul_cap_125.parquet"

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        "Missing processed training data. Run notebook 01 first to create "
        "train_FD001_with_rul_cap_125.parquet."
    )

train_df = pd.read_parquet(INPUT_PATH)
print("Loaded:", INPUT_PATH)
print("Shape:", train_df.shape)
train_df.head(3)

Loaded: /Users/ayushkanyal/Desktop/Predictive Maintainence/data/processed/train_FD001_with_rul_cap_125.parquet
Shape: (20631, 28)


,unit_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,rul_raw,rul
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191,125
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190,125
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189,125


In [4]:
drop_sensors = set(RECOMMENDED_DROPPED_SENSORS)
sensor_cols = [col for col in get_sensor_columns(train_df) if col not in drop_sensors]

rolling_windows = (5, 15, 30)
lag_steps = (1, 3, 5)
sequence_length = 30
sequence_stride = 1

print("Dropped sensors:", sorted(drop_sensors))
print("Kept sensor count:", len(sensor_cols))
print("Rolling windows:", rolling_windows)
print("Lag steps:", lag_steps)
print("Sequence length:", sequence_length)
print("Sequence stride:", sequence_stride)

Dropped sensors: ['sensor_1', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19', 'sensor_5', 'sensor_6']
Kept sensor count: 14
Rolling windows: (5, 15, 30)
Lag steps: (1, 3, 5)
Sequence length: 30
Sequence stride: 1


In [5]:
rolling_df = build_rolling_features(
    train_df,
    sensor_columns=sensor_cols,
    windows=rolling_windows,
    lags=lag_steps,
    min_periods=1,
    drop_na=True,
    include_original_sensors=False,
)

rolling_feature_cols = [col for col in rolling_df.columns if "_roll_" in col or "_lag_" in col]
print("Rolling dataset shape:", rolling_df.shape)
print("Engineered feature count:", len(rolling_feature_cols))
rolling_df.head(3)

/Users/ayushkanyal/Desktop/Predictive Maintainence/src/features/feature_engineering.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[lag_col] = sensor_group.shift(lag)
/Users/ayushkanyal/Desktop/Predictive Maintainence/src/features/feature_engineering.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[mean_col] = sensor_group.transform(
/Users/ayushkanyal/Desktop/Predictive Maintainence/src/features/feature_engineering.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling

Rolling dataset shape: (20131, 140)
Engineered feature count: 126


,unit_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_5,sensor_6,sensor_10,sensor_16,sensor_18,sensor_19,rul_raw,rul,sensor_2_roll_mean_w5,sensor_2_roll_std_w5,sensor_2_roll_mean_w15,sensor_2_roll_std_w15,sensor_2_roll_mean_w30,sensor_2_roll_std_w30,sensor_2_lag_1,sensor_2_lag_3,sensor_2_lag_5,sensor_3_roll_mean_w5,sensor_3_roll_std_w5,sensor_3_roll_mean_w15,sensor_3_roll_std_w15,sensor_3_roll_mean_w30,sensor_3_roll_std_w30,sensor_3_lag_1,sensor_3_lag_3,sensor_3_lag_5,sensor_4_roll_mean_w5,sensor_4_roll_std_w5,sensor_4_roll_mean_w15,sensor_4_roll_std_w15,sensor_4_roll_mean_w30,sensor_4_roll_std_w30,sensor_4_lag_1,sensor_4_lag_3,sensor_4_lag_5,sensor_7_roll_mean_w5,sensor_7_roll_std_w5,sensor_7_roll_mean_w15,sensor_7_roll_std_w15,sensor_7_roll_mean_w30,sensor_7_roll_std_w30,sensor_7_lag_1,sensor_7_lag_3,sensor_7_lag_5,sensor_8_roll_mean_w5,sensor_8_roll_std_w5,sensor_8_roll_mean_w15,sensor_8_roll_std_w15,sensor_8_roll_mean_w30,sensor_8_roll_std_w30,sensor_8_lag_1,sensor_8_lag_3,sensor_8_lag_5,sensor_9_roll_mean_w5,...,sensor_12_roll_std_w15,sensor_12_roll_mean_w30,sensor_12_roll_std_w30,sensor_12_lag_1,sensor_12_lag_3,sensor_12_lag_5,sensor_13_roll_mean_w5,sensor_13_roll_std_w5,sensor_13_roll_mean_w15,sensor_13_roll_std_w15,sensor_13_roll_mean_w30,sensor_13_roll_std_w30,sensor_13_lag_1,sensor_13_lag_3,sensor_13_lag_5,sensor_14_roll_mean_w5,sensor_14_roll_std_w5,sensor_14_roll_mean_w15,sensor_14_roll_std_w15,sensor_14_roll_mean_w30,sensor_14_roll_std_w30,sensor_14_lag_1,sensor_14_lag_3,sensor_14_lag_5,sensor_15_roll_mean_w5,sensor_15_roll_std_w5,sensor_15_roll_mean_w15,sensor_15_roll_std_w15,sensor_15_roll_mean_w30,sensor_15_roll_std_w30,sensor_15_lag_1,sensor_15_lag_3,sensor_15_lag_5,sensor_17_roll_mean_w5,sensor_17_roll_std_w5,sensor_17_roll_mean_w15,sensor_17_roll_std_w15,sensor_17_roll_mean_w30,sensor_17_roll_std_w30,sensor_17_lag_1,sensor_17_lag_3,sensor_17_lag_5,sensor_20_roll_mean_w5,sensor_20_roll_std_w5,sensor_20_roll_mean_w15,sensor_20_roll_std_w15,sensor_20_roll_mean_w30,sensor_20_roll_std_w30,sensor_20_lag_1,sensor_20_lag_3,sensor_20_lag_5,sensor_21_roll_mean_w5,sensor_21_roll_std_w5,sensor_21_roll_mean_w15,sensor_21_roll_std_w15,sensor_21_roll_mean_w30,sensor_21_roll_std_w30,sensor_21_lag_1,sensor_21_lag_3,sensor_21_lag_5
0,1,6,-0.0043,-0.0001,100.0,518.67,14.62,21.61,1.3,0.03,2388,100.0,186,125,642.264,0.114822,642.190000,0.195874,642.190000,0.195874,642.37,642.35,641.82,1585.984,3.475593,1586.603333,3.461838,1586.603333,3.461838,1582.85,1587.99,1589.70,1402.760,2.617396,1402.400000,2.521303,1402.400000,2.521303,1406.22,1404.20,1400.60,554.226,0.324382,554.248333,0.300301,554.248333,0.300301,554.00,554.26,554.36,2388.062,0.031241,2388.061667,0.028529,2388.061667,0.028529,2388.06,2388.08,2388.06,9050.264,...,0.418426,522.181667,0.418426,522.19,522.42,521.66,2388.050,0.020976,2388.045000,0.022174,2388.045000,0.022174,2388.04,2388.03,2388.02,8133.040,0.857485,8133.970000,2.221989,8133.970000,2.221989,8133.80,8133.23,8138.62,8.41160,0.023011,8.412917,0.021211,8.412917,0.021211,8.4294,8.4178,8.4195,391.6,1.019804,391.666667,0.942809,391.666667,0.942809,393.0,390.0,392.0,38.942,0.045782,38.961667,0.060668,38.961667,0.060668,38.90,38.95,39.06,23.38260,0.028118,23.388667,0.029032,23.388667,0.029032,23.4044,23.3442,23.4190
1,1,7,0.0010,0.0001,100.0,518.67,14.62,21.61,1.3,0.03,2388,100.0,185,125,642.330,0.124740,642.231429,0.207807,642.231429,0.207807,642.10,642.35,642.15,1586.084,3.645126,1587.420000,3.778087,1587.420000,3.778087,1584.47,1582.79,1591.82,1401.686,3.263192,1401.738571,2.841435,1401.738571,2.841435,1398.37,1401.87,1403.14,554.344,0.220418,554.261429,0.279869,554.261429,0.279869,554.67,554.45,553.75,2388.058,0.034871,2388.055714,0.030170,2388.055714,0.030170,2388.02,2388.11,2388.04,9053.276,...,0.390400,522.201429,0.390400,521.68,522.86,522.28,2388.042,0.019391,2388.042857,0.021189,2388.042857,0.021189,2388.03,2388.08,2388.07,8133.206,0.575243,8133.734286,2.136652,8133.734286,2.136652,8132.85,81

In [6]:
sequence_feature_cols = ["op_setting_1", "op_setting_2", "op_setting_3", *sensor_cols]
x_seq, y_seq, seq_index = create_sequence_dataset(
    train_df,
    feature_columns=sequence_feature_cols,
    target_col="rul",
    sequence_length=sequence_length,
    stride=sequence_stride,
)

print("X sequence shape:", x_seq.shape)
print("y sequence shape:", y_seq.shape)
seq_index.head()

X sequence shape: (17731, 30, 17)
y sequence shape: (17731,)


,unit_id,cycle,rul
0,1,30,125.0
1,1,31,125.0
2,1,32,125.0
3,1,33,125.0
4,1,34,125.0


In [7]:
tracking_dir = PROJECT_ROOT / "mlruns"
tracking_dir.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(tracking_dir.as_uri())
mlflow.set_experiment("cmapss_phase2_feature_engineering")

data_hash = dataframe_fingerprint(train_df, columns=["unit_id", "cycle", "rul", *sensor_cols])

with mlflow.start_run(run_name=f"FD001_phase2_roll_seq_t{sequence_length}") as run:
    mlflow.log_param("dataset_id", "FD001")
    mlflow.log_param("rul_cap", 125)
    mlflow.log_param("dropped_sensors", ",".join(sorted(drop_sensors)))
    mlflow.log_param("kept_sensor_count", len(sensor_cols))
    mlflow.log_param("rolling_windows", ",".join(map(str, rolling_windows)))
    mlflow.log_param("lag_steps", ",".join(map(str, lag_steps)))
    mlflow.log_param("sequence_length", sequence_length)
    mlflow.log_param("sequence_stride", sequence_stride)
    mlflow.log_param("data_fingerprint_sha256", data_hash)

    mlflow.log_metric("train_rows", float(len(train_df)))
    mlflow.log_metric("rolling_rows", float(len(rolling_df)))
    mlflow.log_metric("rolling_feature_count", float(len(rolling_feature_cols)))
    mlflow.log_metric("sequence_samples", float(x_seq.shape[0]))
    mlflow.log_metric("sequence_timesteps", float(x_seq.shape[1]))
    mlflow.log_metric("sequence_features", float(x_seq.shape[2]))

    mlflow.log_text("\n".join(rolling_feature_cols), "feature_columns/rolling_feature_columns.txt")

run_id = run.info.run_id
print("MLflow run_id:", run_id)
print("MLflow tracking URI:", mlflow.get_tracking_uri())

/Users/ayushkanyal/Desktop/Predictive Maintainence/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/24 19:41:53 INFO mlflow.tracking.fluent: Experiment with name 'cmapss_phase2_feature_engineering' does not exist. Creating a new experiment.


MLflow run_id: 6aeba8a3a08f4879b9897e90450af875
MLflow tracking URI: file:///Users/ayushkanyal/Desktop/Predictive%20Maintainence/mlruns


In [8]:
rolling_output = PROCESSED_DIR / "train_FD001_rolling_features.parquet"
x_output = PROCESSED_DIR / f"train_FD001_sequences_x_t{sequence_length}.npy"
y_output = PROCESSED_DIR / f"train_FD001_sequences_y_t{sequence_length}.npy"
index_output = PROCESSED_DIR / f"train_FD001_sequence_index_t{sequence_length}.parquet"

rolling_df.to_parquet(rolling_output, index=False)
np.save(x_output, x_seq)
np.save(y_output, y_seq)
seq_index.to_parquet(index_output, index=False)

print("Saved:", rolling_output)
print("Saved:", x_output)
print("Saved:", y_output)
print("Saved:", index_output)

Saved: /Users/ayushkanyal/Desktop/Predictive Maintainence/data/processed/train_FD001_rolling_features.parquet
Saved: /Users/ayushkanyal/Desktop/Predictive Maintainence/data/processed/train_FD001_sequences_x_t30.npy
Saved: /Users/ayushkanyal/Desktop/Predictive Maintainence/data/processed/train_FD001_sequences_y_t30.npy
Saved: /Users/ayushkanyal/Desktop/Predictive Maintainence/data/processed/train_FD001_sequence_index_t30.parquet
